In [24]:
import networkx as nx
import numpy as np
import pandas as pd

In [25]:
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set directory path to your uploaded dataset location
data_dir = "/content/drive/MyDrive/AML_Dataset"
# 3. Import temporal split logic from split_data.py
if '/content' not in sys.path:
    sys.path.append('/content')

from split_data import temporal_split

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Load the dataset using the Google Drive path
print("Loading dataset...")
csv_path = os.path.join(data_dir, 'HI-Small_Trans.csv')
df = pd.read_csv(csv_path)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

Loading dataset...


In [27]:
import gc

# 1. Apply temporal split FIRST
train_df, test_df = temporal_split(df)


# Function to run memory-optimized feature engineering
def engineer_features(data):
    # Downcast floats/ints to reduce RAM footprint by ~50%
    df_out = data.copy()
    for col in df_out.select_dtypes(include=["float64"]).columns:
        df_out[col] = df_out[col].astype("float32")
    for col in df_out.select_dtypes(include=["int64"]).columns:
        df_out[col] = df_out[col].astype("int32")

    # --- Basic Flags & Domain Indicators ---
    df_out["is_cross_border"] = (
        df_out["From Bank"] != df_out["To Bank"]
    ).astype("int8")
    df_out["is_cross_currency"] = (
        df_out["Receiving Currency"] != df_out["Payment Currency"]
    ).astype("int8")
    df_out["is_round_amount"] = (df_out["Amount Paid"] % 1000 == 0).astype(
        "int8"
    )

    df_out["is_ach"] = (df_out["Payment Format"] == "ACH").astype("int8")
    df_out["is_ach_sar"] = (
        (df_out["Payment Format"] == "ACH")
        & (
            (df_out["Receiving Currency"] == "Saudi Riyal")
            | (df_out["Payment Currency"] == "Saudi Riyal")
        )
    ).astype("int8")

    # --- Account Aggregations ---
    tx_counts = (
        df_out.groupby("Account")
        .size()
        .reset_index(name="tx_count_out")
        .astype({"tx_count_out": "int32"})
    )
    df_out = df_out.merge(tx_counts, on="Account", how="left")
    del tx_counts

    unique_counterparties = (
        df_out.groupby("Account")["Account.1"]
        .nunique()
        .reset_index(name="unique_counterparties_out")
        .astype({"unique_counterparties_out": "int32"})
    )
    df_out = df_out.merge(unique_counterparties, on="Account", how="left")
    del unique_counterparties

    # --- Memory-Optimized Rolling Time Spikes ---
    df_out = df_out.sort_values("Timestamp")

    # Grouping directly without creating heavy intermediate indexed DataFrames
    for window in ["1h", "24h"]:
        rolling_series = (
            df_out.groupby("Account")
            .rolling(window, on="Timestamp")["Amount Paid"]
            .count()
            .reset_index(drop=True)
            .astype("float32")
        )
        df_out[f"tx_{window}_burst"] = rolling_series
        del rolling_series
        gc.collect()

    # --- Graph Topology & Network Centrality ---
    G = nx.from_pandas_edgelist(
        df_out,
        source="Account",
        target="Account.1",
        create_using=nx.DiGraph(),
    )
    pagerank_scores = nx.pagerank(G, alpha=0.85)
    in_degrees = dict(G.in_degree())
    out_degrees = dict(G.out_degree())

    df_out["account_pagerank"] = (
        df_out["Account"].map(pagerank_scores).fillna(0).astype("float32")
    )
    df_out["counterparty_pagerank"] = (
        df_out["Account.1"].map(pagerank_scores).fillna(0).astype("float32")
    )
    df_out["account_in_degree"] = (
        df_out["Account"].map(in_degrees).fillna(0).astype("int32")
    )
    df_out["account_out_degree"] = (
        df_out["Account"].map(out_degrees).fillna(0).astype("int32")
    )
    del G, pagerank_scores, in_degrees, out_degrees
    gc.collect()

    # --- In/Out Volume Ratios ---
    account_totals = (
        df_out.groupby("Account")
        .agg(
            total_paid=("Amount Paid", "sum"),
            total_received=("Amount Received", "sum"),
        )
        .reset_index()
    )
    account_totals["in_out_ratio"] = (
        account_totals["total_received"] / (account_totals["total_paid"] + 1)
    ).astype("float32")
    df_out = df_out.merge(
        account_totals[["Account", "in_out_ratio"]], on="Account", how="left"
    )
    del account_totals

    df_out.fillna(0, inplace=True)
    gc.collect()
    return df_out


# Execute feature pipeline cleanly
train_engineered = engineer_features(train_df)
del train_df
gc.collect()

test_engineered = engineer_features(test_df)
del test_df
gc.collect()

# Export leakage-free datasets
train_engineered.to_parquet(
    os.path.join(data_dir, "train_features.parquet"), index=False
)
test_engineered.to_parquet(
    os.path.join(data_dir, "test_features.parquet"), index=False
)
print("Leakage-free feature tables successfully saved to Drive.")

Leakage-free feature tables successfully saved to Drive.
